# 06 — Evaluation and error analysis (Tasks 5 and 6)

Analyses the saved validation predictions of one run (no retraining). With several folds trained, the predictions of
all folds are combined into out-of-fold (OOF) predictions. Both validation views are analysed:

* **val_clean** — validation images with plain preprocessing
* **val_shift** — the same images after ShiftSim (deterministic per image id, see notebook 02)

Contents: overall metrics; per-class precision / recall / F1; confusion matrices; most frequent confusions;
species-level and part-level accuracy; accuracy by image size and by aspect ratio; ShiftSim-perturbed vs unchanged
images; confidence and calibration; the highest-loss validation images; learning curves; the experiments table; and
validation vs leaderboard scores of all submissions. Figures and tables only.

In [ ]:
RUN = "convnext_tiny_shift"   # run to analyse
FOLDS = "all"                 # "all" = every fold with saved predictions, or e.g. "0" or "0,1"
USE_TTA = False               # True = analyse TTA predictions (created by notebook 07)
N_WORST = 36                  # images in the highest-loss grids

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))  # the notebooks live in the repo root, next to src/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid")
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

from sklearn.metrics import classification_report, confusion_matrix, f1_score

from src.config import CLASSES, EXPERIMENTS_CSV, LOG_DIR, PRED_DIR, SUBMISSIONS_CSV, load_shiftsim_params, make_dirs
from src.data import FaceDataset, load_folds_df
from src.plots import plot_confusion, plot_history, savefig, show_grid
from src.transforms import ShiftSim
from src.utils import load_preds, pred_path, softmax

make_dirs()
if FOLDS == "all":
    folds = sorted(int(p.name[4]) for p in (PRED_DIR / RUN).glob("fold*_val_clean.npz"))
else:
    folds = [int(f) for f in FOLDS.split(",")]
print("run:", RUN, "| folds:", folds, "| TTA:", USE_TTA)

## 1. Load predictions

One table per split with, for every validation image: true class, predicted class, probability of the true class,
highest probability, cross-entropy loss, original size and the size of the image as fed to the model
(for val_shift: after ShiftSim).

In [ ]:
meta = load_folds_df().set_index("image_id")
sim = ShiftSim(**load_shiftsim_params())

tables = {}
for split in ["val_clean", "val_shift"]:
    parts = []
    for f in folds:
        ids, logits, labels = load_preds(pred_path(RUN, f, split, USE_TTA))
        p = softmax(logits)
        parts.append(pd.DataFrame({"image_id": ids, "fold": f, "label": labels, "pred": p.argmax(1),
                                   "p_true": p[np.arange(len(labels)), labels], "p_max": p.max(1)}))
        for k, c in enumerate(CLASSES):
            parts[-1][f"p_{c}"] = p[:, k]
    t = pd.concat(parts, ignore_index=True)
    t["correct"] = t["label"] == t["pred"]
    t["loss"] = -np.log(np.clip(t["p_true"], 1e-7, 1))
    t["true"] = np.array(CLASSES)[t["label"]]
    t["predicted"] = np.array(CLASSES)[t["pred"]]
    t["w"], t["h"] = meta.loc[t["image_id"], "w"].values, meta.loc[t["image_id"], "h"].values
    t["path"] = meta.loc[t["image_id"], "path"].values
    if split == "val_shift":
        t["fed_w"], t["fed_h"] = sim.simulate_sizes(t["w"].values, t["h"].values, keys=t["image_id"].astype(int).values)
        t["perturbed"] = (t["fed_w"] != t["w"]) | (t["fed_h"] != t["h"])
    else:
        t["fed_w"], t["fed_h"], t["perturbed"] = t["w"], t["h"], False
    t["fed_short"] = np.minimum(t["fed_w"], t["fed_h"])
    t["fed_aspect"] = t["fed_w"] / t["fed_h"]
    tables[split] = t
print({k: len(v) for k, v in tables.items()})

## 2. Overall metrics

In [ ]:
overall = pd.DataFrame({split: {
    "images": len(t), "accuracy": t["correct"].mean(), "macro F1": f1_score(t["label"], t["pred"], average="macro"),
    "log loss": t["loss"].mean(), "mean max-probability": t["p_max"].mean()} for split, t in tables.items()})
display(overall.round(4))
if len(folds) > 1:
    display(pd.DataFrame({s: t.groupby("fold")["correct"].mean() for s, t in tables.items()}).round(4)
            .rename(columns=lambda c: c + " accuracy"))

## 3. Per-class precision, recall and F1

In [ ]:
reports = {split: pd.DataFrame(classification_report(t["label"], t["pred"], labels=range(len(CLASSES)), target_names=CLASSES,
                                                      output_dict=True, zero_division=0)).T
           for split, t in tables.items()}
report = pd.concat(reports, axis=1)
display(report.round(4))
report.to_csv(Path("outputs/figures") / f"06_class_report_{RUN}.csv")

fig, ax = plt.subplots(figsize=(11, 3.8))
f1 = pd.DataFrame({s: r.loc[CLASSES, "f1-score"] for s, r in reports.items()})
f1.plot.bar(ax=ax)
ax.set_ylim(max(0, f1.values.min() - 0.05), 1)
ax.set_title(f"F1 per class ({RUN})")
plt.tight_layout()
savefig(f"06_f1_per_class_{RUN}")
plt.show()

> **Observations (to be written by me):**

## 4. Confusion matrices and most frequent confusions

In [ ]:
for split, t in tables.items():
    cm = confusion_matrix(t["label"], t["pred"], labels=range(len(CLASSES)))
    fig, axes = plt.subplots(1, 2, figsize=(16, 6.5))
    plot_confusion(cm, CLASSES, title=f"{split}", normalize=False, ax=axes[0])
    plot_confusion(cm, CLASSES, title=f"{split}", normalize=True, ax=axes[1])
    plt.tight_layout()
    savefig(f"06_confusion_{RUN}_{split}")
    plt.show()

In [ ]:
conf = {}
for split, t in tables.items():
    wrong = t[~t["correct"]]
    conf[split] = wrong.groupby(["true", "predicted"]).size().sort_values(ascending=False)
top_conf = pd.concat({s: c.rename("count") for s, c in conf.items()}, axis=1).fillna(0).astype(int)
top_conf["share of all errors (val_shift) %"] = 100 * top_conf["val_shift"] / top_conf["val_shift"].sum()
display(top_conf.sort_values("val_shift", ascending=False).head(15).round(2))

Accuracy after collapsing the 8 classes to **species** (Cat / Dog) and to **part** (Eye / Face / Mouth / Nose).

In [ ]:
rows = {}
for split, t in tables.items():
    rows[split] = {
        "8-class accuracy": t["correct"].mean(),
        "species accuracy (Cat/Dog)": (t["true"].str[:3] == t["predicted"].str[:3]).mean(),
        "part accuracy (Eye/Face/Mouth/Nose)": (t["true"].str[3:] == t["predicted"].str[3:]).mean(),
        "errors with right species, wrong part %": 100 * ((t["true"].str[:3] == t["predicted"].str[:3]) & ~t["correct"]).sum() / max(1, (~t["correct"]).sum()),
        "errors with right part, wrong species %": 100 * ((t["true"].str[3:] == t["predicted"].str[3:]) & ~t["correct"]).sum() / max(1, (~t["correct"]).sum()),
    }
display(pd.DataFrame(rows).round(4))

> **Observations (to be written by me):**

## 5. Accuracy by image size and aspect ratio

Size and aspect refer to the image **as fed to the model** (for val_shift: after ShiftSim). Counts are shown
under each bar group.

In [ ]:
size_bins = [0, 16, 24, 32, 48, 64, 96, 128, 256, 100_000]
aspect_bins = [0, 0.8, 0.995, 1.03, 1.3, 1.5, 100]
fig, axes = plt.subplots(1, 2, figsize=(17, 4.5))
for ax, (col, bins, label) in zip(axes, [("fed_short", size_bins, "shorter side (px)"), ("fed_aspect", aspect_bins, "aspect ratio w/h")]):
    frames = []
    for split, t in tables.items():
        g = t.groupby(pd.cut(t[col], bins), observed=False)["correct"].agg(["mean", "size"]).reset_index()
        g["split"] = split
        frames.append(g)
    g = pd.concat(frames)
    g["bucket"] = g[col].astype(str) + "\n" + g.groupby(col, observed=False)["size"].transform(lambda s: "n=" + "/".join(map(str, s)))
    sns.barplot(data=g, x="bucket", y="mean", hue="split", ax=ax)
    ax.set_ylim(max(0, g["mean"].min() - 0.05), 1)
    ax.set_xlabel(label + " (n = clean/shift)")
    ax.set_ylabel("accuracy")
    ax.tick_params(axis="x", labelsize=7)
plt.tight_layout()
savefig(f"06_acc_by_size_aspect_{RUN}")
plt.show()

In [ ]:
t = tables["val_shift"]
heat = t.pivot_table(index="true", columns=pd.cut(t["fed_short"], size_bins), values="correct", aggfunc="mean", observed=False)
fig, ax = plt.subplots(figsize=(12, 4.5))
sns.heatmap(heat.reindex(CLASSES) * 100, annot=True, fmt=".1f", cmap="RdYlGn", vmin=heat.min().min() * 100, vmax=100, ax=ax)
ax.set_title("val_shift accuracy (%) by true class and shorter side")
plt.tight_layout()
savefig(f"06_acc_class_x_size_{RUN}")
plt.show()

display(t.groupby("perturbed")["correct"].agg(["mean", "size"]).rename(index={False: "unchanged by ShiftSim", True: "perturbed by ShiftSim"}).round(4))

> **Observations (to be written by me):**

## 6. Confidence and calibration

Left: distribution of the highest predicted probability for correct and wrong predictions. Right: reliability
diagram — predictions grouped into 10 confidence bins; a perfectly calibrated model lies on the diagonal.
ECE = expected calibration error (bin-size weighted mean |accuracy − confidence|).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for split, t in tables.items():
    sns.histplot(data=t, x="p_max", hue="correct", bins=40, stat="density", common_norm=False, element="step",
                 fill=False, ax=axes[0])
axes[0].set_title("Highest probability: correct vs wrong (both splits)")
eces = {}
for split, t in tables.items():
    b = pd.cut(t["p_max"], np.linspace(0, 1, 11))
    rel = t.groupby(b, observed=True).agg(conf=("p_max", "mean"), acc=("correct", "mean"), n=("correct", "size"))
    eces[split] = float((rel["n"] * (rel["acc"] - rel["conf"]).abs()).sum() / rel["n"].sum())
    axes[1].plot(rel["conf"], rel["acc"], marker="o", label=f"{split} (ECE {eces[split]:.4f})")
axes[1].plot([0, 1], [0, 1], "k--", lw=0.8)
axes[1].set_xlabel("mean confidence in bin")
axes[1].set_ylabel("accuracy in bin")
axes[1].set_title("Reliability diagram")
axes[1].legend()
plt.tight_layout()
savefig(f"06_calibration_{RUN}")
plt.show()

> **Observations (to be written by me):**

## 7. Highest-loss validation images

The images with the largest cross-entropy loss, shown exactly as they entered the transform (val_shift: after
ShiftSim). Title: id, true → predicted, probability of the predicted class.

In [ ]:
for split, t in tables.items():
    worst = t.sort_values("loss", ascending=False).head(N_WORST)
    ds = FaceDataset(worst.assign(label=worst["label"]), transform=None, shift_sim=sim if split == "val_shift" else None)
    imgs = [ds.load(i) for i in range(len(worst))]
    titles = [f"{r.image_id} {r.true}\n→ {r.predicted} {r.p_max:.2f}" for r in worst.itertuples()]
    show_grid(imgs, titles, ncols=12, size=1.5, suptitle=f"Highest-loss {split} images ({RUN})",
              name=f"06_worst_{RUN}_{split}", fontsize=6)
worst_table = tables["val_shift"].sort_values("loss", ascending=False).head(N_WORST)
display(worst_table[["image_id", "fold", "true", "predicted", "p_max", "p_true", "w", "h", "fed_w", "fed_h", "perturbed"]])

> **Observations (to be written by me):**

## 8. Learning curves of the analysed folds

In [ ]:
for f in folds:
    path = LOG_DIR / RUN / f"fold{f}_history.csv"
    if path.exists():
        plot_history(pd.read_csv(path), title=f"{RUN} fold {f}", name=f"06_curves_{RUN}_fold{f}")

## 9. All experiments

In [ ]:
ex = pd.read_csv(EXPERIMENTS_CSV)
display(ex.sort_values(["fold", "val_shift_acc"], ascending=[True, False]))
cv = ex.groupby("run").agg(folds=("fold", "nunique"), val_clean_mean=("val_clean_acc", "mean"),
                           val_shift_mean=("val_shift_acc", "mean"), minutes_total=("minutes", "sum"))
display(cv.sort_values("val_shift_mean", ascending=False).round(4))

## 10. Validation vs leaderboard

All submissions from `results/submissions.csv`. The `lb_public` column is filled in by hand after each upload.

In [ ]:
subs = pd.read_csv(SUBMISSIONS_CSV)
display(subs)
scored = subs.dropna(subset=["lb_public"])
if len(scored):
    scored = scored.assign(gap_clean=scored["val_clean"] - scored["lb_public"], gap_shift=scored["val_shift"] - scored["lb_public"])
    display(scored[["file", "val_clean", "val_shift", "lb_public", "gap_clean", "gap_shift"]].round(4))
    fig, ax = plt.subplots(figsize=(6.5, 5))
    ax.scatter(scored["val_clean"], scored["lb_public"], label="val_clean")
    ax.scatter(scored["val_shift"], scored["lb_public"], label="val_shift")
    for r in scored.itertuples():
        ax.annotate(r.file.replace(".csv", ""), (r.val_shift, r.lb_public), fontsize=7)
    lims = [min(scored[["val_clean", "val_shift", "lb_public"]].min()) - 0.01, 1]
    ax.plot(lims, lims, "k--", lw=0.8)
    ax.set_xlabel("validation accuracy")
    ax.set_ylabel("public leaderboard accuracy")
    ax.legend()
    plt.tight_layout()
    savefig("06_val_vs_lb")
    plt.show()
else:
    print("no leaderboard scores entered yet in results/submissions.csv")

> **Observations (to be written by me):**